In [1]:
import requests
import pandas as pd
import mysql.connector
import random
from datetime import datetime

# -----------------------------
# 1. DOWNLOAD API DATA
# -----------------------------
resource_id = "eed22efa-133e-4e7d-8e18-1ff3156a67b6"

all_records = []
offset = 0
limit = 1000

while True:
    url = f"https://data.opencity.in/api/action/datastore_search?resource_id={resource_id}&limit={limit}&offset={offset}"
    response = requests.get(url).json()
    records = response['result']['records']

    if not records:
        break

    all_records.extend(records)
    offset += limit

print("Total records:", len(all_records))

df = pd.DataFrame(all_records)

# -----------------------------
# 2. CLEAN COLUMN NAMES
# -----------------------------
df.columns = df.columns.str.lower().str.replace(" ", "_")

df.rename(columns={
    "division_mobile_no.": "division_mobile_no",
    "has_cash_counter_?": "has_cash_counter"
}, inplace=True)

# -----------------------------
# 3. DATE FIELDS
# -----------------------------
df["data_date"] = datetime.today().date()
df["system_entry_time"] = datetime.now()

# -----------------------------
# 4. FIX CASH COUNTER
# -----------------------------
df["has_cash_counter"] = df["has_cash_counter"].astype(str).str.upper()

df["has_cash_counter"] = df["has_cash_counter"].replace({
    "Y": "YES",
    "N": "NO",
    "TRUE": "YES",
    "FALSE": "NO"
})

# -----------------------------
# 5. ADD POPULATION
# -----------------------------
df["zone_population"] = [random.randint(50000, 300000) for _ in range(len(df))]

# -----------------------------
# 6. WATER METRICS (FIXED TYPES)
# -----------------------------
df["water_supply_capacity_liters"] = [random.randint(8000000, 20000000) for _ in range(len(df))]

df["daily_water_supply_liters"] = (
    df["water_supply_capacity_liters"] * random.uniform(0.7, 0.9)
).astype(int)

df["daily_water_consumption_liters"] = (
    df["daily_water_supply_liters"] * random.uniform(0.85, 1.05)
).astype(int)

df["water_shortage_liters"] = (
    df["daily_water_consumption_liters"] - df["daily_water_supply_liters"]
).astype(int)

df["pipeline_leakage_percent"] = [round(random.uniform(5, 12), 2) for _ in range(len(df))]
df["reservoir_level_percent"] = [round(random.uniform(40, 95), 2) for _ in range(len(df))]

# -----------------------------
# 7. KPI CALCULATIONS
# -----------------------------
df["per_capita_water_liters"] = df["daily_water_consumption_liters"] / df["zone_population"]
df["supply_demand_ratio"] = df["daily_water_supply_liters"] / df["daily_water_consumption_liters"]

df["consumption_rank"] = df["daily_water_consumption_liters"]\
    .rank(ascending=False)\
    .astype(int)

# -----------------------------
# 8. MYSQL CONNECTION
# -----------------------------
conn = mysql.connector.connect(
    host="127.0.0.1",
    port=3306,
    user="root",
    password="2711",
    database="smart_city"
)

cursor = conn.cursor()

# -----------------------------
# 9. INSERT DATA
# -----------------------------
insert_query = """
INSERT INTO water_chennai_analytics (
data_date,
system_entry_time,
division_no,
zone_no,
division_office_address,
division_office_landline,
division_mobile_no,
has_cash_counter,
zonal_office_address,
zonal_office_landline,
zone_population,
water_supply_capacity_liters,
daily_water_supply_liters,
daily_water_consumption_liters,
water_shortage_liters,
pipeline_leakage_percent,
reservoir_level_percent,
per_capita_water_liters,
supply_demand_ratio,
consumption_rank
)
VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)
"""

data = df[[
    "data_date",
    "system_entry_time",
    "division_no",
    "zone_no",
    "division_office_address",
    "division_office_landline",
    "division_mobile_no",
    "has_cash_counter",
    "zonal_office_address",
    "zonal_office_landline",
    "zone_population",
    "water_supply_capacity_liters",
    "daily_water_supply_liters",
    "daily_water_consumption_liters",
    "water_shortage_liters",
    "pipeline_leakage_percent",
    "reservoir_level_percent",
    "per_capita_water_liters",
    "supply_demand_ratio",
    "consumption_rank"
]].values.tolist()

cursor.executemany(insert_query, data)

conn.commit()

print("Data inserted successfully")

cursor.close()
conn.close()

Total records: 200
Data inserted successfully
